# 01 — Introduction au Reinforcement Learning avec Stable-Baselines3

Ce notebook couvre les bases du RL : environnement, agent, modèle, entraînement et évaluation.

Source : https://pythonprogramming.net/introduction-reinforcement-learning-stable-baselines-3-tutorial/

## 1. Concepts fondamentaux

Le RL repose sur trois éléments :

| Élément | Rôle |
|---------|------|
| **Environnement** | Le problème à résoudre (ex: LunarLander) |
| **Agent** | L'entité qui prend des décisions |
| **Modèle (algorithme)** | La stratégie d'apprentissage (PPO, A2C, SAC…) |

### Boucle d'interaction

```
Agent → Action → Environnement → Observation + Reward → Agent
```

L'agent apprend à maximiser la somme des récompenses cumulées au fil du temps.

## 2. Explorer l'environnement LunarLander-v3

Avant d'entraîner quoi que ce soit, il faut comprendre l'espace d'observation et d'action.

In [ ]:
import gymnasium as gym

# Créer l'environnement (render_mode passé à make(), pas à render())
env = gym.make('LunarLander-v3')

# Nouvelle API : reset() retourne (observation, info)
obs, info = env.reset()

print("=== Espace d'observation ===")
print(f"Shape : {env.observation_space.shape}")
print(f"Exemple : {obs}")

print("\n=== Espace d'action ===")
print(f"Type : {env.action_space}")
print(f"Exemple d'action aléatoire : {env.action_space.sample()}")

### Décodage des 8 observations de LunarLander

| Index | Signification |
|-------|---------------|
| 0 | Position x |
| 1 | Position y |
| 2 | Vitesse x |
| 3 | Vitesse y |
| 4 | Angle |
| 5 | Vitesse angulaire |
| 6 | Contact pied gauche (booléen) |
| 7 | Contact pied droit (booléen) |

### Actions disponibles (espace Discret)

| Action | Effet |
|--------|-------|
| 0 | Rien |
| 1 | Moteur gauche |
| 2 | Moteur principal |
| 3 | Moteur droit |

**Discret vs Continu** : LunarLander a un espace discret (classification). D'autres environnements comme Pendulum ont des espaces continus (régression).

## 3. Agent aléatoire (baseline)

Avant d'entraîner, observons les performances d'un agent qui agit aléatoirement.

In [ ]:
env = gym.make('LunarLander-v3')
obs, info = env.reset()

total_reward = 0
steps = 0

while True:
    action = env.action_space.sample()  # action aléatoire
    
    # Nouvelle API : step() retourne 5 valeurs
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    steps += 1
    
    done = terminated or truncated
    if done:
        break

print(f"Steps : {steps}")
print(f"Reward total : {total_reward:.2f}")
env.close()

## 4. Choisir un algorithme compatible

Stable-Baselines3 supporte plusieurs algorithmes, avec des compatibilités différentes :

| Algorithme | Action Discrète | Action Continue | Remarque |
|-----------|:-:|:-:|----------|
| PPO | ✅ | ✅ | Recommandé, robuste |
| A2C | ✅ | ✅ | Plus volatile |
| SAC | ❌ | ✅ | Continue uniquement |
| DDPG | ❌ | ✅ | Continue uniquement |
| DQN | ✅ | ❌ | Discrète uniquement |

LunarLander-v3 a un espace discret → PPO, A2C ou DQN.

## 5. Entraînement avec A2C

In [ ]:
from stable_baselines3 import A2C

env = gym.make('LunarLander-v3')

# MlpPolicy = réseau de neurones dense (adapté aux observations vectorielles)
model = A2C('MlpPolicy', env, verbose=1)

# 10 000 pas d'entraînement (court, pour explorer)
model.learn(total_timesteps=10_000)

print("Entraînement A2C (10K steps) terminé")

### Lire les métriques pendant l'entraînement

- **ep_rew_mean** : récompense moyenne par épisode (à maximiser)
- **ep_len_mean** : longueur moyenne des épisodes
- **value_loss** : erreur du réseau de valeur (doit diminuer)
- **policy_gradient_loss** : erreur de la politique

## 6. Évaluation du modèle entraîné

In [ ]:
from stable_baselines3.common.evaluation import evaluate_policy

# evaluate_policy fait tourner N épisodes et retourne mean/std reward
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10)
print(f"A2C (10K steps) — Reward moyen : {mean_reward:.2f} ± {std_reward:.2f}")

## 7. Entraînement avec PPO (plus de steps)

In [ ]:
from stable_baselines3 import PPO

env = gym.make('LunarLander-v3')

model_ppo = PPO('MlpPolicy', env, verbose=1)
model_ppo.learn(total_timesteps=100_000)

mean_reward, std_reward = evaluate_policy(model_ppo, env, n_eval_episodes=10)
print(f"PPO (100K steps) — Reward moyen : {mean_reward:.2f} ± {std_reward:.2f}")

## 8. Visualiser l'agent en action

Pour visualiser, créer un environnement avec `render_mode='human'`.

In [ ]:
# render_mode='human' ouvre une fenêtre graphique
env_render = gym.make('LunarLander-v3', render_mode='human')
obs, info = env_render.reset()

for step in range(500):
    action, _states = model_ppo.predict(obs)
    obs, reward, terminated, truncated, info = env_render.step(action)
    
    if terminated or truncated:
        obs, info = env_render.reset()

env_render.close()
print("Épisode de visualisation terminé")

## Résumé

| Concept | Détail |
|---------|--------|
| `gym.make(env, render_mode=...)` | Créer l'environnement |
| `env.reset()` | Retourne `(obs, info)` |
| `env.step(action)` | Retourne `(obs, reward, terminated, truncated, info)` |
| `done = terminated or truncated` | Fin d'épisode |
| `model.learn(total_timesteps=N)` | Entraîner le modèle |
| `model.predict(obs)` | Obtenir l'action recommandée |
| `evaluate_policy(model, env, n_eval_episodes=N)` | Évaluer les performances |

**Prochaine étape** : sauvegarder et charger les modèles → notebook `02_save_load_tensorboard.ipynb`